In [0]:
from pyspark.sql.functions import current_timestamp, col , to_date
from pyspark.sql.functions import col, to_date, sum as spark_sum, current_date
from pyspark.sql.functions import current_date, lit
from pyspark.sql.window import Window

In [0]:
base_path = "/Volumes/de_workspace26/tejpal_shop/tejpal_raw_volume/"

In [0]:

customers = spark.table("de_workspace26.tejpal_shop.tejpal_bronze_customers")
orders = spark.table("de_workspace26.tejpal_shop.tejpal_bronze_orders")
products = spark.table("de_workspace26.tejpal_shop.tejpal_bronze_products")


customers_clean = customers \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"]) \
    .withColumn("signup_date", to_date("signup_date"))


orders_clean = orders \
    .dropDuplicates(["order_id"]) \
    .dropna(subset=["order_id", "customer_id", "product_id"]) \
    .withColumn("order_date", to_date("order_date")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .filter((col("quantity") > 0) & (col("unit_price") > 0)) \
    .withColumn("revenue", col("quantity") * col("unit_price"))

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders_clean = orders_clean.withColumn(
    "cumulative_revenue",
    spark_sum("revenue").over(window_spec)
)

products_clean = products \
    .dropDuplicates(["product_id"]) \
    .dropna(subset=["product_id"])


In [0]:
silver_df = orders_clean \
    .join(customers_clean, "customer_id", "left") \
    .join(products_clean, "product_id", "left") \
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "unit_price",
        "revenue",
        "cumulative_revenue",
        "city",
        "loyalty_tier",
        "product_name",
        "category",
        "region"
    )

silver_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_silver_orders")


In [0]:

display(spark.table("de_workspace26.tejpal_shop.tejpal_silver_orders"))

In [0]:

customers_silver = (
    customers_clean
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

customers_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_silver_customers")

In [0]:
display(spark.table("de_workspace26.tejpal_shop.tejpal_silver_customers"))

In [0]:
## SCD TYPE 2

In [0]:
from pyspark.sql.functions import current_date, lit, to_date
from delta.tables import DeltaTable


updates_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(base_path + "customers_update.csv")

updates_clean = (
    updates_df
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", to_date("signup_date"))
)


existing = DeltaTable.forName(spark, "de_workspace26.tejpal_shop.tejpal_silver_customers")


existing.alias("existing").merge(
    updates_clean.alias("updates"),
    """existing.customer_id = updates.customer_id 
       AND existing.is_current = true 
       AND existing.loyalty_tier <> updates.loyalty_tier"""
).whenMatchedUpdate(set={
    "is_current":         "false",
    "effective_end_date": "current_date()"
}).execute()


new_records = (
    updates_clean.alias("s")
    .join(
        spark.table("de_workspace26.tejpal_shop.tejpal_silver_customers")
             .filter("is_current = false")
             .alias("t"),
        "customer_id"
    )
    .filter("t.loyalty_tier <> s.loyalty_tier")
    .select("s.*")
    .dropDuplicates(["customer_id"])
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date",   lit(None).cast("date"))
    .withColumn("is_current",           lit(True))
)

new_records.write.format("delta") \
    .mode("append") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_silver_customers")

print("SCD Type 2 MERGE complete.")

In [0]:
# 4. Verify — each customer_id should have at most ONE is_current = true row
display(
    spark.table("de_workspace26.tejpal_shop.tejpal_silver_customers")
    .orderBy("customer_id", "effective_start_date")
)

In [0]:
%sql
SELECT customer_id, COUNT(*) as cnt
FROM de_workspace26.tejpal_shop.tejpal_silver_customers
WHERE is_current = true
GROUP BY customer_id
HAVING COUNT(*) > 1;


In [0]:
%sql
ALTER TABLE de_workspace26.tejpal_shop.tejpal_silver_orders
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
-- Cell 1: Find which version CDF was enabled at
DESCRIBE HISTORY de_workspace26.tejpal_shop.tejpal_silver_orders;

In [0]:
%sql
SELECT * 
FROM table_changes(
  'de_workspace26.tejpal_shop.tejpal_silver_orders',
  3
);

In [0]:
%sql
OPTIMIZE de_workspace26.tejpal_shop.tejpal_silver_orders
ZORDER BY (customer_id, order_date);